# 🏛️ Fine-Tuning Qwen 2.5 1.5B Instruct for Public Budget (SEFA-PR / UEL)
**Project:** UEL Public Budget SLM — LABGEP / NIGEP  
**Hardware:** Google Colab (Free Tesla T4 GPU)  
**Objective:** Train the QLoRA adapter and export the quantized GGUF (`Q4_K_M`) binary for CPU execution in Ollama.

### 1. Install Optimized Dependencies (Unsloth)

In [ ]:
%%capture
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade --no-cache-dir trl peft accelerate bitsandbytes


### 2. Load Qwen 2.5 1.5B Base Model (4-bit NF4)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# Inject LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

### 3. Upload and Format Dataset (1,500 ChatML Pairs)
Upload `dataset_budget_qwen_1500.jsonl` using the Files sidebar in Google Colab.

In [ ]:
from datasets import load_dataset

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

dataset = load_dataset("json", data_files="dataset_budget_qwen_1500.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"Total training examples loaded: {len(dataset)}")

### 4. Supervised Fine-Tuning (`SFTTrainer`)

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,
        warmup_steps = 10,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "paged_adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 42,
        output_dir = "outputs",
        report_to = "none"
    ),
)

trainer_stats = trainer.train()

### 5. Direct Export to Quantized GGUF Binary (`Q4_K_M`) for Ollama

In [ ]:
# Merge LoRA weights and export quantized GGUF binary
model.save_pretrained_gguf("qwen2.5_budget_1.5b_gguf", tokenizer, quantization_method = "q4_k_m")

!ls -lh qwen2.5_budget_1.5b_gguf

### 6. Save GGUF Binary to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copy the generated GGUF binary to Google Drive
!mkdir -p /content/drive/MyDrive/SLM_Modelos/
!find /content -name "*.gguf" -exec cp {} /content/drive/MyDrive/SLM_Modelos/unsloth.Q4_K_M.gguf \;
!ls -lh /content/drive/MyDrive/SLM_Modelos/
print("GGUF binary successfully copied to Google Drive!")